# Positional Encodings

Transformers are permutation-equivariant without positional information. This notebook implements and compares the major positional encoding schemes.

**Interview questions:**
- "Why do transformers need positional encoding?"
- "RoPE vs sinusoidal — what's the difference?"
- "How does ALiBi work and why is it good for extrapolation?"

---

## Summary Table

| Method | Type | Learned? | Extrapolation | Used in |
|--------|------|----------|---------------|----------|
| Sinusoidal | Absolute | No | Poor | Original Transformer |
| Learned | Absolute | Yes | Poor | GPT-2, BERT |
| RoPE | Relative (rotation) | No | Moderate (with NTK-aware scaling) | LLaMA, Mistral, Qwen |
| ALiBi | Relative (bias) | No | Good | BLOOM, MPT |

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)

## 1. Sinusoidal Positional Encoding (Vaswani et al., 2017)

$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d}}\right), \quad PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d}}\right)$$

Key properties:
- Deterministic, no learned parameters
- Each dimension is a sinusoid with different frequency
- Relative positions can be represented as linear functions: $PE_{pos+k}$ is a linear function of $PE_{pos}$
- Poor extrapolation beyond training length

In [ ]:
class SinusoidalPositionalEncoding(nn.Module):
    """Original Vaswani et al. positional encoding.
    Added to token embeddings: x = x + PE
    """
    def __init__(self, d_model: int, max_len: int = 5000, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)  # (max_len, 1)
        # Compute div_term in log space for numerical stability
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )  # (d_model/2,)
        
        pe[:, 0::2] = torch.sin(position * div_term)  # even dimensions
        pe[:, 1::2] = torch.cos(position * div_term)  # odd dimensions
        
        self.register_buffer('pe', pe.unsqueeze(0))  # (1, max_len, d_model)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (batch, seq_len, d_model)"""
        return self.dropout(x + self.pe[:, :x.size(1)])

# Create and visualize
sin_pe = SinusoidalPositionalEncoding(d_model=128, dropout=0.0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Heatmap of positional encodings
pe_matrix = sin_pe.pe[0, :100, :].numpy()
im = axes[0].imshow(pe_matrix, aspect='auto', cmap='RdBu', vmin=-1, vmax=1)
axes[0].set_xlabel('Dimension')
axes[0].set_ylabel('Position')
axes[0].set_title('Sinusoidal PE Heatmap')
plt.colorbar(im, ax=axes[0])

# Individual dimension frequencies
positions = np.arange(100)
for dim in [0, 10, 30, 60, 100, 126]:
    axes[1].plot(positions, pe_matrix[:, dim], label=f'dim {dim}', alpha=0.7)
axes[1].set_xlabel('Position')
axes[1].set_ylabel('Value')
axes[1].set_title('PE values at different dimensions')
axes[1].legend()

plt.tight_layout()
plt.show()

print("Low dimensions = high frequency (changes fast with position)")
print("High dimensions = low frequency (changes slowly with position)")

In [ ]:
# Show dot-product similarity between positions (should decay with distance)
pe_vecs = sin_pe.pe[0, :64, :]  # (64, 128)
similarity = torch.mm(pe_vecs, pe_vecs.T)  # (64, 64)

plt.figure(figsize=(6, 5))
plt.imshow(similarity.numpy(), cmap='RdBu', vmin=-40, vmax=40)
plt.colorbar(label='Dot product')
plt.xlabel('Position')
plt.ylabel('Position')
plt.title('Sinusoidal PE: Position Dot-Product Similarity')
plt.tight_layout()
plt.show()

## 2. Learned Positional Embeddings (GPT-2, BERT)

Simply learn a position embedding table. Simple, effective, but:
- Cannot extrapolate beyond max trained length
- Adds `max_len * d_model` parameters

In [ ]:
class LearnedPositionalEmbedding(nn.Module):
    """Standard learned positional embedding (GPT-2, BERT style)."""
    def __init__(self, d_model: int, max_len: int = 512, dropout: float = 0.1):
        super().__init__()
        self.pos_embedding = nn.Embedding(max_len, d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (batch, seq_len, d_model)"""
        seq_len = x.size(1)
        positions = torch.arange(seq_len, device=x.device).unsqueeze(0)  # (1, seq_len)
        return self.dropout(x + self.pos_embedding(positions))

# Test
learned_pe = LearnedPositionalEmbedding(d_model=128, max_len=512)
x = torch.randn(2, 10, 128)
out = learned_pe(x)
print(f"Output shape: {out.shape}")
print(f"Parameters: {sum(p.numel() for p in learned_pe.parameters()):,}")
print(f"(That's {512 * 128:,} = max_len * d_model)")

## 3. Rotary Position Embedding (RoPE) — Su et al., 2021

The key insight: encode position by **rotating** the query and key vectors.

For a 2D subspace at frequency $\theta_i$:
$$\begin{pmatrix} q_0' \\ q_1' \end{pmatrix} = \begin{pmatrix} \cos(m\theta_i) & -\sin(m\theta_i) \\ \sin(m\theta_i) & \cos(m\theta_i) \end{pmatrix} \begin{pmatrix} q_0 \\ q_1 \end{pmatrix}$$

Where $m$ is position, $\theta_i = 10000^{-2i/d}$.

**Why it's elegant:**
- $\langle \text{RoPE}(q, m), \text{RoPE}(k, n) \rangle$ depends only on $q$, $k$, and $(m-n)$
- Encodes **relative** position through the inner product
- Applied to Q, K only (not V) — V carries content, not position
- No extra parameters

In [ ]:
class RotaryPositionalEmbedding(nn.Module):
    """RoPE: Rotary Position Embedding.
    
    Applied to Q, K tensors of shape (batch, n_heads, seq_len, head_dim).
    head_dim must be even (we pair consecutive dimensions for 2D rotations).
    """
    def __init__(self, head_dim: int, max_len: int = 4096, base: float = 10000.0):
        super().__init__()
        assert head_dim % 2 == 0, "head_dim must be even for RoPE"
        
        self.head_dim = head_dim
        self.base = base
        
        # Precompute frequency for each dimension pair
        # theta_i = base^(-2i/d) for i in [0, d/2)
        inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
        self.register_buffer('inv_freq', inv_freq)  # (head_dim / 2,)
        
        # Precompute sin/cos for positions [0, max_len)
        self._build_cache(max_len)
    
    def _build_cache(self, max_len: int):
        positions = torch.arange(max_len, dtype=torch.float)
        # Outer product: (max_len, head_dim/2)
        freqs = torch.outer(positions, self.inv_freq)
        # Duplicate for pairing: (max_len, head_dim)
        emb = torch.cat([freqs, freqs], dim=-1)
        self.register_buffer('cos_cached', emb.cos(), persistent=False)
        self.register_buffer('sin_cached', emb.sin(), persistent=False)
    
    @staticmethod
    def _rotate_half(x: torch.Tensor) -> torch.Tensor:
        """Rotate pairs: [x0, x1, x2, x3, ...] -> [-x_{d/2}, ..., x0, x1, ...]
        This is the efficient implementation of the rotation matrix.
        """
        x1 = x[..., :x.shape[-1] // 2]
        x2 = x[..., x.shape[-1] // 2:]
        return torch.cat([-x2, x1], dim=-1)
    
    def forward(self, q: torch.Tensor, k: torch.Tensor, 
                offset: int = 0) -> tuple[torch.Tensor, torch.Tensor]:
        """Apply RoPE to query and key.
        
        Args:
            q, k: (batch, n_heads, seq_len, head_dim)
            offset: position offset (for KV-cache during generation)
        """
        seq_len = q.size(2)
        cos = self.cos_cached[offset:offset + seq_len].unsqueeze(0).unsqueeze(0)
        sin = self.sin_cached[offset:offset + seq_len].unsqueeze(0).unsqueeze(0)
        
        # RoPE formula: x' = x * cos(m*theta) + rotate_half(x) * sin(m*theta)
        q_rotated = q * cos + self._rotate_half(q) * sin
        k_rotated = k * cos + self._rotate_half(k) * sin
        
        return q_rotated, k_rotated

# Test
rope = RotaryPositionalEmbedding(head_dim=64)
q = torch.randn(2, 8, 16, 64)
k = torch.randn(2, 8, 16, 64)
q_rot, k_rot = rope(q, k)
print(f"Q shape: {q_rot.shape}")  # unchanged
print(f"No learned parameters: {sum(p.numel() for p in rope.parameters())}")

In [ ]:
# Verify the key property: dot product depends only on relative position
rope_test = RotaryPositionalEmbedding(head_dim=64)

# Create identical q, k vectors at different positions
q_base = torch.randn(1, 1, 1, 64).expand(1, 1, 32, 64).clone()
k_base = torch.randn(1, 1, 1, 64).expand(1, 1, 32, 64).clone()

q_rot, k_rot = rope_test(q_base, k_base)

# Dot product at positions (5, 10) should equal (15, 20) — same relative distance of 5
dot_5_10 = (q_rot[0, 0, 5] * k_rot[0, 0, 10]).sum().item()
dot_15_20 = (q_rot[0, 0, 15] * k_rot[0, 0, 20]).sum().item()
dot_0_5 = (q_rot[0, 0, 0] * k_rot[0, 0, 5]).sum().item()
dot_0_10 = (q_rot[0, 0, 0] * k_rot[0, 0, 10]).sum().item()

print(f"Same relative distance (5):  pos(5,10)={dot_5_10:.4f}, pos(15,20)={dot_15_20:.4f}, pos(0,5)={dot_0_5:.4f}")
print(f"Different relative distance: pos(0,10)={dot_0_10:.4f}")
print("\nSame-distance dot products should be equal (they are!).")

In [ ]:
# Visualize RoPE frequencies
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Frequency spectrum
dims = torch.arange(0, 64, 2)
freqs = 1.0 / (10000.0 ** (dims.float() / 64))
axes[0].semilogy(dims.numpy(), freqs.numpy(), 'b.-')
axes[0].set_xlabel('Dimension pair index')
axes[0].set_ylabel('Frequency (theta_i)')
axes[0].set_title('RoPE Frequencies (log scale)')
axes[0].grid(True, alpha=0.3)

# Dot product decay with relative position
q_fixed = torch.randn(1, 1, 1, 64)
k_fixed = torch.randn(1, 1, 1, 64)

# Compute dot product at various relative distances
max_dist = 128
dots = []
for dist in range(max_dist):
    q_at_0 = q_fixed.expand(1, 1, dist + 1, 64).clone()
    k_at_0 = k_fixed.expand(1, 1, dist + 1, 64).clone()
    q_r, k_r = rope_test(q_at_0, k_at_0)
    dot = (q_r[0, 0, 0] * k_r[0, 0, dist]).sum().item()
    dots.append(dot)

axes[1].plot(dots)
axes[1].set_xlabel('Relative distance')
axes[1].set_ylabel('Dot product')
axes[1].set_title('RoPE: Dot Product vs Relative Distance')
axes[1].axhline(y=0, color='gray', linestyle='--', alpha=0.5)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. ALiBi (Attention with Linear Biases) — Press et al., 2022

Instead of adding positional info to embeddings, ALiBi adds a **position-dependent bias directly to attention scores**:

$$\text{softmax}\left(\frac{QK^T}{\sqrt{d_k}} + m \cdot \text{bias}\right)$$

Where `bias[i,j] = -|i - j|` (negative of distance) and `m` is a head-specific slope.

Slopes are geometric: $m_i = 2^{-8i/H}$ for head $i$ of $H$ total.

**Why it extrapolates well:** the linear bias is the same functional form at any length. No frequencies to go out of range.

In [ ]:
class ALiBi(nn.Module):
    """Attention with Linear Biases.
    
    Returns a bias tensor to be ADDED to attention scores.
    No modification to embeddings — purely modifies attention logits.
    """
    def __init__(self, n_heads: int, max_len: int = 4096):
        super().__init__()
        self.n_heads = n_heads
        
        # Compute slopes: geometric sequence
        # For n_heads=8: [1/2, 1/4, 1/8, 1/16, 1/32, 1/64, 1/128, 1/256]
        slopes = self._get_slopes(n_heads)
        self.register_buffer('slopes', torch.tensor(slopes).float())
        
        # Precompute distance matrix
        positions = torch.arange(max_len)
        # distance[i, j] = -|i - j|
        distance = -(positions.unsqueeze(0) - positions.unsqueeze(1)).abs().float()
        self.register_buffer('distance', distance)
    
    @staticmethod
    def _get_slopes(n_heads: int) -> list:
        """Get the slopes for each head.
        
        If n_heads is a power of 2: slopes = 2^(-8*i/n_heads) for i in 1..n_heads
        Otherwise: use the closest power of 2 and interpolate.
        """
        def _get_slopes_power_of_2(n):
            start = 2 ** (-(2 ** -(math.log2(n) - 3)))
            return [start * (start ** i) for i in range(n)]
        
        if math.log2(n_heads).is_integer():
            return _get_slopes_power_of_2(n_heads)
        else:
            # Not a power of 2: use closest power and interleave
            closest_pow2 = 2 ** math.floor(math.log2(n_heads))
            base_slopes = _get_slopes_power_of_2(closest_pow2)
            extra_slopes = _get_slopes_power_of_2(2 * closest_pow2)
            extra_slopes = extra_slopes[0::2][:n_heads - closest_pow2]
            return base_slopes + extra_slopes
    
    def forward(self, seq_len: int) -> torch.Tensor:
        """Returns bias of shape (1, n_heads, seq_len, seq_len)."""
        # slopes: (n_heads,) -> (1, n_heads, 1, 1)
        # distance: (seq_len, seq_len)
        bias = self.slopes.view(1, -1, 1, 1) * self.distance[:seq_len, :seq_len].unsqueeze(0).unsqueeze(0)
        return bias  # (1, n_heads, seq_len, seq_len)

# Test
alibi = ALiBi(n_heads=8)
bias = alibi(16)  # (1, 8, 16, 16)
print(f"ALiBi bias shape: {bias.shape}")
print(f"Slopes: {alibi.slopes.tolist()}")
print(f"No learnable parameters: {sum(p.numel() for p in alibi.parameters())}")

In [ ]:
# Visualize ALiBi biases per head
alibi = ALiBi(n_heads=8)
bias = alibi(32)  # (1, 8, 32, 32)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i, ax in enumerate(axes.flat):
    im = ax.imshow(bias[0, i].numpy(), cmap='RdBu', aspect='auto')
    ax.set_title(f'Head {i} (slope={alibi.slopes[i]:.4f})')
    ax.set_xlabel('Key pos')
    ax.set_ylabel('Query pos')
    plt.colorbar(im, ax=ax)

plt.suptitle('ALiBi: Attention Biases per Head\n(blue = discouraged, red = encouraged)', fontsize=13)
plt.tight_layout()
plt.show()

print("Steep slopes (head 0) = very local attention")
print("Gentle slopes (head 7) = broader attention window")

## 5. Comparison: Attention Patterns

Let's see how each method shapes the attention distribution for a toy input.

In [ ]:
# Compare attention patterns from each method
d_model = 64
n_heads = 4
head_dim = d_model // n_heads
seq_len = 32
batch = 1

# Random content embeddings (same for all methods)
torch.manual_seed(42)
x = torch.randn(batch, seq_len, d_model)

# Projection weights (shared)
W_q = nn.Linear(d_model, d_model, bias=False)
W_k = nn.Linear(d_model, d_model, bias=False)

def compute_attention_scores(q, k, bias=None):
    """Return softmax attention weights."""
    scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(head_dim)
    if bias is not None:
        scores = scores + bias
    return F.softmax(scores, dim=-1)

q = W_q(x).view(batch, seq_len, n_heads, head_dim).transpose(1, 2)
k = W_k(x).view(batch, seq_len, n_heads, head_dim).transpose(1, 2)

# 1. No positional encoding
attn_none = compute_attention_scores(q, k)

# 2. Sinusoidal (add to embeddings before projection)
sin_pe = SinusoidalPositionalEncoding(d_model, dropout=0.0)
x_sin = sin_pe(x)
q_sin = W_q(x_sin).view(batch, seq_len, n_heads, head_dim).transpose(1, 2)
k_sin = W_k(x_sin).view(batch, seq_len, n_heads, head_dim).transpose(1, 2)
attn_sin = compute_attention_scores(q_sin, k_sin)

# 3. RoPE (apply rotation to Q, K after projection)
rope = RotaryPositionalEmbedding(head_dim)
q_rope, k_rope = rope(q, k)
attn_rope = compute_attention_scores(q_rope, k_rope)

# 4. ALiBi (add bias to attention scores)
alibi = ALiBi(n_heads=n_heads)
alibi_bias = alibi(seq_len)  # (1, n_heads, seq_len, seq_len)
attn_alibi = compute_attention_scores(q, k, bias=alibi_bias)

# Visualize head 0 for each method
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
titles = ['No Position', 'Sinusoidal', 'RoPE', 'ALiBi']
attns = [attn_none, attn_sin, attn_rope, attn_alibi]

for ax, title, attn in zip(axes, titles, attns):
    im = ax.imshow(attn[0, 0].detach().numpy(), aspect='auto', cmap='viridis')
    ax.set_title(title)
    ax.set_xlabel('Key position')
    ax.set_ylabel('Query position')
    plt.colorbar(im, ax=ax)

plt.suptitle('Attention Patterns (Head 0)', fontsize=13)
plt.tight_layout()
plt.show()

## 6. Extrapolation Comparison

How well does each method handle sequences longer than training length?

In [ ]:
# Simulate extrapolation: train on length 64, test on length 256
train_len = 64
test_len = 256

# For sinusoidal: PE is defined for any length, but values seen during training are limited
# For learned: cannot go beyond max_len at all
# For RoPE: frequencies are the same, but longer positions = more rotations
# For ALiBi: bias grows linearly with distance — always well-defined

print("Extrapolation behavior summary:")
print("="*60)
print(f"{'Method':<15} {'Can extrapolate?':<20} {'Quality'}")
print("-"*60)
print(f"{'Sinusoidal':<15} {'Yes (defined)':<20} {'Poor — model never learned to use these PE values'}")
print(f"{'Learned':<15} {'No (hard limit)':<20} {'N/A — crashes or needs extension'}")
print(f"{'RoPE':<15} {'Partially':<20} {'Degrades; NTK-aware scaling helps significantly'}")
print(f"{'ALiBi':<15} {'Yes (natural)':<20} {'Good — linear bias naturally extends'}")

# Visualize: attention entropy (uncertainty) at different sequence lengths
def attention_entropy(attn_weights):
    """Compute per-query attention entropy."""
    # Clamp to avoid log(0)
    return -(attn_weights * (attn_weights + 1e-10).log()).sum(dim=-1).mean().item()

lengths = [16, 32, 64, 128, 256, 512]
entropies = {'RoPE': [], 'ALiBi': [], 'Sinusoidal': []}

for L in lengths:
    x = torch.randn(1, L, d_model)
    q = W_q(x).view(1, L, n_heads, head_dim).transpose(1, 2)
    k = W_k(x).view(1, L, n_heads, head_dim).transpose(1, 2)
    
    # RoPE
    rope_ext = RotaryPositionalEmbedding(head_dim, max_len=1024)
    q_r, k_r = rope_ext(q, k)
    attn_r = compute_attention_scores(q_r, k_r)
    entropies['RoPE'].append(attention_entropy(attn_r))
    
    # ALiBi
    alibi_ext = ALiBi(n_heads, max_len=1024)
    bias = alibi_ext(L)
    attn_a = compute_attention_scores(q, k, bias=bias)
    entropies['ALiBi'].append(attention_entropy(attn_a))
    
    # Sinusoidal
    sin_pe_ext = SinusoidalPositionalEncoding(d_model, max_len=1024, dropout=0.0)
    x_s = sin_pe_ext(x)
    q_s = W_q(x_s).view(1, L, n_heads, head_dim).transpose(1, 2)
    k_s = W_k(x_s).view(1, L, n_heads, head_dim).transpose(1, 2)
    attn_s = compute_attention_scores(q_s, k_s)
    entropies['Sinusoidal'].append(attention_entropy(attn_s))

plt.figure(figsize=(8, 4))
for name, ent in entropies.items():
    plt.plot(lengths, ent, 'o-', label=name)
plt.axvline(x=64, color='red', linestyle='--', alpha=0.5, label='Train length')
plt.xlabel('Sequence Length')
plt.ylabel('Attention Entropy')
plt.title('Attention Entropy vs Sequence Length (higher = more uniform)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. NTK-Aware RoPE Scaling

To extend RoPE to longer contexts, we can scale the base frequency:

$$\theta'_i = \left(\text{base} \cdot \alpha^{d/(d-2)}\right)^{-2i/d}$$

where $\alpha = \text{new\_length} / \text{original\_length}$.

This scales high frequencies less than low frequencies (Neural Tangent Kernel-aware scaling).

In [ ]:
class NTKAwareRoPE(RotaryPositionalEmbedding):
    """RoPE with NTK-aware dynamic scaling for context extension."""
    def __init__(self, head_dim, max_len=4096, base=10000.0, scale_factor=1.0):
        # Override parent init to use scaled frequencies
        nn.Module.__init__(self)
        self.head_dim = head_dim
        self.base = base
        
        # NTK-aware scaling
        scaled_base = base * (scale_factor ** (head_dim / (head_dim - 2)))
        inv_freq = 1.0 / (scaled_base ** (torch.arange(0, head_dim, 2).float() / head_dim))
        self.register_buffer('inv_freq', inv_freq)
        self._build_cache(max_len)

# Compare frequencies
rope_base = RotaryPositionalEmbedding(head_dim=64, max_len=4096)
rope_ntk = NTKAwareRoPE(head_dim=64, max_len=16384, scale_factor=4.0)

plt.figure(figsize=(8, 4))
plt.semilogy(rope_base.inv_freq.numpy(), 'b.-', label='Original (4K context)')
plt.semilogy(rope_ntk.inv_freq.numpy(), 'r.-', label='NTK-scaled (16K context, 4x)')
plt.xlabel('Dimension pair index')
plt.ylabel('Frequency (log scale)')
plt.title('RoPE Frequencies: Original vs NTK-Scaled')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("NTK scaling: high frequencies (low dims) barely change, low frequencies (high dims) are scaled down.")
print("This preserves local position resolution while extending the range.")

## Interview Notes

**"Why do transformers need positional encoding?"**
- Self-attention is permutation-equivariant: `Attention(Pi*X) = Pi*Attention(X)` for any permutation Pi
- Without positional info, the model can't distinguish "the cat sat" from "sat cat the"
- Positional encoding breaks this symmetry

**"RoPE vs sinusoidal?"**
- Sinusoidal: additive, absolute positions added to embeddings before attention
- RoPE: multiplicative (rotation), applied to Q/K after projection, encodes relative position through dot product
- RoPE is strictly more expressive: attention score depends on content AND relative position
- Sinusoidal: relative position info gets entangled with content after projection

**"How does ALiBi work?"**
- Adds `slope * (-|query_pos - key_pos|)` to attention logits (before softmax)
- Different heads use different slopes (geometric sequence)
- Steep slopes = local attention, gentle slopes = broad attention
- No modification to embeddings, just attention scores
- Best extrapolation because the bias is always well-defined for any distance

**"Which should I use?"**
- 2024-2025 consensus: RoPE (with NTK scaling or YaRN for extension) is the dominant choice
- Used in LLaMA, Mistral, Qwen, Gemma
- ALiBi is good but less common now (BLOOM, MPT)
- Learned: only BERT/GPT-2 era, fixed context window